# 05 — SVM, KNN, Naive Bayes

Three classic, still widely-used model families: margin-based, instance-based, and probabilistic.

## Support Vector Machines: Classification

In [ ]:
from sklearn.svm import SVC, LinearSVC

svc = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True)   # probability=True enables predict_proba (slower fit)
svc.fit(X_train, y_train)
svc.predict(X_test)
svc.support_vectors_       # the support vectors that define the decision boundary

linear_svc = LinearSVC(C=1.0, max_iter=5000)   # faster than SVC(kernel="linear") for large datasets
linear_svc.fit(X_train, y_train)


## SVM Kernels

In [ ]:
SVC(kernel="linear")             # linear decision boundary
SVC(kernel="poly", degree=3)     # polynomial boundary
SVC(kernel="rbf", gamma=0.1)     # Gaussian/RBF — most common nonlinear kernel
SVC(kernel="sigmoid")            # sigmoid kernel, less common


## Support Vector Regression

In [ ]:
from sklearn.svm import SVR

svr = SVR(kernel="rbf", C=1.0, epsilon=0.1)   # epsilon = margin of tolerance, no penalty within it
svr.fit(X_train, y_train)


## K-Nearest Neighbors

In [ ]:
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

knn = KNeighborsClassifier(n_neighbors=5, weights="distance", metric="minkowski", p=2)  # p=2 -> euclidean
knn.fit(X_train, y_train)
knn.predict(X_test)
knn.kneighbors(X_test[:1], n_neighbors=5)   # (distances, indices) of nearest neighbors

knn_reg = KNeighborsRegressor(n_neighbors=5)
knn_reg.fit(X_train, y_train)


## Choosing k via Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score

scores = []
k_range = range(1, 31)
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    s = cross_val_score(knn, X_train, y_train, cv=5, scoring="accuracy").mean()
    scores.append(s)

plt.plot(k_range, scores)
plt.xlabel("k")
plt.ylabel("CV accuracy")
plt.show()


## Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, ComplementNB

gnb = GaussianNB()                    # continuous features, assumes Gaussian distribution per class
gnb.fit(X_train, y_train)

mnb = MultinomialNB(alpha=1.0)        # discrete counts, e.g. word counts in text classification
mnb.fit(X_train_counts, y_train)      # alpha = Laplace/Lidstone smoothing

bnb = BernoulliNB()                   # binary/boolean features
cnb = ComplementNB()                  # designed for imbalanced text classification


## Naive Bayes for Text Classification (typical pipeline)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import make_pipeline

text_clf = make_pipeline(
    TfidfVectorizer(stop_words="english", max_features=5000),
    MultinomialNB(alpha=0.1),
)
text_clf.fit(text_train, y_train)
text_clf.predict(["a new document to classify"])


## Common Gotchas
- SVMs are sensitive to feature scale — always scale features before fitting (unlike trees).
- `SVC` scales poorly beyond ~10k-50k samples (training is O(n^2) to O(n^3)) — use `LinearSVC` or `SGDClassifier(loss="hinge")` for large linear problems.
- KNN has no real "training" — all cost is at prediction time; also suffers from the curse of dimensionality in high-dimensional feature spaces.
- `GaussianNB` assumes independent, normally-distributed features per class — often violated but the classifier is surprisingly robust anyway.